# MLflow → FAIRSCAPE RO-Crate

You tracked an experiment in MLflow. MLflow already knows the parameters, the
metrics, which dataset went in, which artifacts and models came out, and who
ran it — that is most of a provenance record, already written down. This
notebook turns it into a FAIRSCAPE **EVI RO-Crate** without re-entering any of
it.

End to end, run top to bottom:

1. train a small scikit-learn model, logging it to a local `mlruns/` store
   (a parent run and a nested child run, a logged dataset input, artifacts,
   and a model);
2. convert that store into `crate/ro-crate-metadata.json`, copying the
   artifacts and the model in so the crate stands on its own;
3. read the crate back — the graph, the provenance edges, the run in full,
   the schema MLflow's column spec gave us for free;
4. validate it against `fairscape_models`, and check the ARKs are
   deterministic.

**Requirements**

```bash
pip install mlflow scikit-learn pandas          # to produce the runs
pip install -e ../..                            # fairscape-conversion (or rely on the path shim below)
pip install fairscape-cli                       # optional: schema inference for copied .csv artifacts
```

MLflow has no end-of-run hook — its plugin points are stores and context
providers, none of which fire when a run ends — so this is a *post-hoc*
exporter over the `MlflowClient` read API. That is why it works against any
backend: the `mlruns/` directory used here, `sqlite:///mlflow.db`, or a remote
`http://` tracking server. Nothing about your training code has to change.

> Running this notebook rewrites `mlruns/` and `crate/` next to it. The `crate/`
> checked into the repository is the output of one such run — `git checkout
> examples/mlflow/crate` puts it back.

## Setup

In [1]:
import importlib.util
import json
import os
import shutil
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "mlflow_to_rocrate.ipynb").exists():
    raise SystemExit("run this notebook from the examples/mlflow/ directory")
PACKAGE = HERE.parents[1]                    # .../fairscape_conversion

# Bind `fairscape_conversion` to this checkout so the notebook runs without
# installing anything. `pip install -e ../..` works just as well.
if "fairscape_conversion" not in sys.modules:
    spec = importlib.util.spec_from_file_location(
        "fairscape_conversion", PACKAGE / "__init__.py",
        submodule_search_locations=[str(PACKAGE)])
    module = importlib.util.module_from_spec(spec)
    sys.modules["fairscape_conversion"] = module
    spec.loader.exec_module(module)

import warnings

# Two upstream warnings are noise here and say nothing about the crate:
# MLflow's hint about integer columns in an inferred model signature, and a
# pydantic serializer notice raised by fairscape-cli's schema inference.
warnings.filterwarnings("ignore", category=UserWarning, module="mlflow.types.utils")
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic.main")

try:
    import mlflow
    import mlflow.sklearn
    import pandas as pd
    import sklearn
except ImportError as missing:
    raise SystemExit(f"{missing.name} is needed to produce the runs: "
                     "pip install mlflow scikit-learn pandas")

MLRUNS = HERE / "mlruns"      # the tracking store this notebook creates
CRATE = HERE / "crate"        # where the RO-Crate lands
WORK = HERE / "work"          # scratch files we log as artifacts
WORK.mkdir(exist_ok=True)

print(f"mlflow {mlflow.__version__} | scikit-learn {sklearn.__version__} "
      f"| pandas {pd.__version__}")
print(f"tracking store -> ./{MLRUNS.name}/")
print(f"crate          -> ./{CRATE.name}/")

mlflow 2.22.0 | scikit-learn 1.8.0 | pandas 2.2.3
tracking store -> ./mlruns/
crate          -> ./crate/


## 1. Train something worth describing

Nothing FAIRSCAPE-specific here — this is ordinary MLflow instrumentation, and
it is exactly what the converter reads later:

| what we log | what it becomes in the crate |
|---|---|
| `mlflow.log_input(...)` | an EVI `Dataset` **plus a `Schema`** built from the column spec |
| `log_params` / `log_metrics` | `parameter` and `additionalProperty` on the `Computation` |
| `log_artifact` | an EVI `Dataset` with `generatedBy` pointing back at the run |
| `log_model` | an EVI `Dataset` typed `mls:Model`, carrying its flavor |
| a nested run | a `Computation` that `isPartOf` its parent |
| `mlflow.source.name` (set for you) | an EVI `Software` node for the training script |

In [2]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

shutil.rmtree(MLRUNS, ignore_errors=True)          # start from a clean store
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
mlflow.set_tracking_uri(MLRUNS.as_uri())
mlflow.set_experiment("iris-classifier")

features, target = load_iris(return_X_y=True, as_frame=True)
frame = features.assign(species=target)
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=0, stratify=target)

# MLflow 3 renamed log_model's `artifact_path` to `name`; support both.
model_kwargs = ({"name": "model"} if int(mlflow.__version__.split(".")[0]) >= 3
                else {"artifact_path": "model"})

with mlflow.start_run(run_name="rf-baseline"):
    mlflow.log_input(
        mlflow.data.from_pandas(frame, name="iris", targets="species"),
        context="training")
    mlflow.log_params({"n_estimators": 200, "max_depth": 4, "random_state": 0})

    classifier = RandomForestClassifier(
        n_estimators=200, max_depth=4, random_state=0).fit(X_train, y_train)
    predicted = classifier.predict(X_test)
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, predicted),
        "f1_macro": f1_score(y_test, predicted, average="macro"),
    })

    matrix = WORK / "confusion_matrix.csv"
    pd.DataFrame(confusion_matrix(y_test, predicted)).to_csv(matrix, index=False)
    mlflow.log_artifact(str(matrix))
    mlflow.sklearn.log_model(classifier, input_example=X_train.head(3),
                             **model_kwargs)

    # A nested run: the crate keeps the parent/child relationship as isPartOf.
    with mlflow.start_run(run_name="feature-importance", nested=True):
        importance = pd.DataFrame({"feature": features.columns,
                                   "importance": classifier.feature_importances_})
        ranking = WORK / "feature_importance.csv"
        importance.sort_values("importance", ascending=False).to_csv(
            ranking, index=False)
        mlflow.log_artifact(str(ranking))
        mlflow.log_metric("top_importance", float(importance.importance.max()))

print(f"accuracy {accuracy_score(y_test, predicted):.3f} — 2 runs logged")

2026/09/01 15:42:20 INFO mlflow.tracking.fluent: Experiment with name 'iris-classifier' does not exist. Creating a new experiment.


accuracy 0.947 — 2 runs logged


## 2. Convert the tracking store

One call. It walks the experiment through the `MlflowClient` read API, copies
every artifact and the model directory into `crate/` (one folder per run), and
returns the crate as a dict.

`schemas=True` additionally infers an EVI `Schema` for each copied tabular
artifact via `fairscape-cli`; if that isn't installed the call prints a note
and skips it. The `Schema` for the *logged dataset input* needs no such help —
it comes from the column spec MLflow already stored, so no data file is read.

In [3]:
from fairscape_conversion.plugins import mlflow as mlflow_conversion

shutil.rmtree(CRATE, ignore_errors=True)

crate = mlflow_conversion.convert(
    "import",
    MLRUNS,                          # or "sqlite:///mlflow.db", or "http://…"
    experiment="iris-classifier",    # or run_id="…" for one run + its children
    crate_dir=str(CRATE),            # artifacts and models are copied in here
    naan="59853",                    # your ARK Name Assigning Authority Number
    name="Iris classifier — MLflow experiment",
    description="Random forest trained on the iris dataset, tracked in MLflow "
                "and converted to an EVI RO-Crate.",
    author="Example Researcher",
    keywords="mlflow, iris, random forest, provenance",
    license="https://spdx.org/licenses/CC-BY-4.0",
    schemas=True,                    # infer Schemas for copied .csv artifacts
)

metadata = CRATE / "ro-crate-metadata.json"
metadata.write_text(json.dumps(crate, indent=2, default=str))
print(f"{len(crate['@graph'])} nodes -> {metadata.relative_to(HERE)}")

13 nodes -> crate/ro-crate-metadata.json


## 3. Read the crate back

First the shape of the graph: what got described, and as what.

In [4]:
def short(type_value):
    """Trim the vocabulary IRIs so the @types fit in a column."""
    prefixes = {"https://w3id.org/EVI#": "EVI#", "evi:": "EVI#", "EVI:": "EVI#",
                "https://schema.org/": "", "http://www.w3.org/ns/prov#": "prov:"}
    values = type_value if isinstance(type_value, list) else [type_value]
    out = []
    for value in values:
        text = str(value)
        for prefix, replacement in prefixes.items():
            if text.startswith(prefix):
                text = replacement + text[len(prefix):]
                break
        out.append(text)
    return ", ".join(out)


print(f"{'@type':<32} {'name':<44} @id")
print(f"{'-' * 32} {'-' * 44} {'-' * 34}")
for node in crate["@graph"]:
    print(f"{short(node.get('@type', '')):<32} "
          f"{str(node.get('name', ''))[:44]:<44} {node['@id'][:34]}")

@type                            name                                         @id
-------------------------------- -------------------------------------------- ----------------------------------
CreativeWork                                                                  ro-crate-metadata.json
Dataset, EVI#ROCrate             Iris classifier — MLflow experiment          ark:59853/rocrate-iris-classifier-
prov:Activity, EVI#Computation   MLflow run 'rf-baseline'                     ark:59853/computation-rf-baseline-
prov:Activity, EVI#Computation   MLflow run 'feature-importance'              ark:59853/computation-feature-impo
prov:Entity, EVI#Software        ipykernel_launcher.py                        ark:59853/software-ipykernel-launc
prov:Entity, EVI#Software        MLflow                                       ark:59853/software-mlflow-907d9b5
prov:Entity, EVI#Dataset         iris                                         ark:59853/dataset-iris-a44c0fa
prov:Entity, EVI#Dataset       

Then the edges — the part a list of names can't show. This is the provenance
claim the crate is making: *this run used this dataset and this script, and
produced this model.*

In [5]:
EDGES = ("usedSoftware", "usedDataset", "generatedBy", "generated", "isPartOf")
names = {n["@id"]: n.get("name", n["@id"]) for n in crate["@graph"]}

for node in crate["@graph"]:
    for edge in EDGES:
        references = node.get(edge) or []
        if isinstance(references, dict):
            references = [references]
        for reference in references:
            target = reference.get("@id") if isinstance(reference, dict) else reference
            print(f"{str(names.get(node['@id'], ''))[:42]:<44} "
                  f"--{edge}--> {str(names.get(target, target))[:42]}")

MLflow run 'rf-baseline'                     --usedSoftware--> ipykernel_launcher.py
MLflow run 'rf-baseline'                     --usedSoftware--> MLflow
MLflow run 'rf-baseline'                     --usedDataset--> iris
MLflow run 'rf-baseline'                     --generated--> confusion_matrix.csv
MLflow run 'rf-baseline'                     --generated--> model
MLflow run 'feature-importance'              --usedSoftware--> ipykernel_launcher.py
MLflow run 'feature-importance'              --usedSoftware--> MLflow
MLflow run 'feature-importance'              --generated--> feature_importance.csv
MLflow run 'feature-importance'              --isPartOf--> MLflow run 'rf-baseline'
model                                        --generatedBy--> MLflow run 'rf-baseline'
feature_importance.csv                       --generatedBy--> MLflow run 'feature-importance'
confusion_matrix.csv                         --generatedBy--> MLflow run 'rf-baseline'


## 4. One run, in full

The parent `Computation`. Note what survived the trip that you never wrote
down by hand: the command line, who ran it, the wall-clock window, the
parameters, the metrics as `PropertyValue`s, and `mlflowRunId` so the node
still points back at the run it came from.

> The `Software` node says `ipykernel_launcher.py` because MLflow records the
> notebook kernel as the source. Train from a script and you get that script —
> `train.py`, with its git commit if the working tree is a repository.

In [6]:
run_node = next(node for node in crate["@graph"]
                if "Computation" in short(node.get("@type", ""))
                and not node.get("isPartOf"))
print(json.dumps(run_node, indent=2))

{
  "@id": "ark:59853/computation-rf-baseline-3fd9e47",
  "@type": [
    "prov:Activity",
    "https://w3id.org/EVI#Computation"
  ],
  "name": "MLflow run 'rf-baseline'",
  "description": "MLflow run 'rf-baseline' of experiment 'iris-classifier', reconstructed post-hoc from the MLflow tracking store by mlflow-fairscape",
  "runBy": "justin",
  "dateCreated": "2026-09-01T15:42:20.472000-04:00",
  "command": "python /opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py",
  "usedSoftware": [
    {
      "@id": "ark:59853/software-ipykernel-launcher-py-461b0f9"
    },
    {
      "@id": "ark:59853/software-mlflow-907d9b5"
    }
  ],
  "usedDataset": [
    {
      "@id": "ark:59853/dataset-iris-a44c0fa"
    }
  ],
  "generated": [
    {
      "@id": "ark:59853/dataset-confusion-matrix-csv-8fa7303"
    },
    {
      "@id": "ark:59853/dataset-model-c7bf999"
    }
  ],
  "parameter": [
    "max_depth=4",
    "n_estimators=200",
    "random_state=0"
  ],
  "additionalProperty": [


## 5. The model, and the schema you got for free

The logged model is an EVI `Dataset` additionally typed `mls:Model` — it keeps
its flavor, its size, and a `contentUrl` pointing at the copy inside the crate,
and it is `generatedBy` the run above.

The schema beneath it was built from the column spec MLflow recorded when
`log_input` was called: names, order, types, required-ness. No data file was
opened to produce it.

In [7]:
model_node = next(node for node in crate["@graph"]
                  if node.get("additionalType", "").endswith("#Model"))
print(json.dumps(model_node, indent=2))

schema_node = next(node for node in crate["@graph"]
                   if "Schema" in short(node.get("@type", ""))
                   and "MLflow dataset" in str(node.get("name")))
print(f"\n{schema_node['name']}  ({schema_node['@id']})")
print(f"  {'column':<24} {'type':<10} required")
for column, spec in schema_node["properties"].items():
    print(f"  {column:<24} {spec['type']:<10} "
          f"{column in schema_node.get('required', [])}")

{
  "@id": "ark:59853/dataset-model-c7bf999",
  "@type": [
    "prov:Entity",
    "https://w3id.org/EVI#Dataset"
  ],
  "name": "model",
  "additionalType": "http://www.w3.org/ns/mls#Model",
  "author": "Example Researcher",
  "description": "MLflow model 'model' (sklearn flavor), logged by run 'rf-baseline' with MLflow 2.22.0",
  "datePublished": "2026-09-01T15:42:24.146461-04:00",
  "keywords": [
    "mlflow",
    "iris",
    "random forest",
    "provenance"
  ],
  "format": "sklearn",
  "generatedBy": [
    {
      "@id": "ark:59853/computation-rf-baseline-3fd9e47"
    }
  ],
  "mlflowModelId": "cce96f10b1334967bb3fbb0ff0e02698",
  "contentUrl": "rf-baseline-2dcf075c/model/",
  "contentSize": "284491"
}

Schema for MLflow dataset 'iris'  (ark:59853/schema-iris-e04fad4)
  column                   type       required
  sepal length (cm)        number     True
  sepal width (cm)         number     True
  petal length (cm)        number     True
  petal width (cm)         number     Tr

## 6. Is it a valid crate?

`fairscape-conversion` depends on `fairscape_models`, so the same pydantic
models the FAIRSCAPE services use are already available to check the output.

In [8]:
from fairscape_models.rocrate import ROCrateV1_2

validated = ROCrateV1_2.model_validate(json.loads(metadata.read_text()))
print(f"valid ROCrateV1_2 — {len(validated.metadataGraph)} nodes in @graph")

valid ROCrateV1_2 — 13 nodes in @graph


## 7. The ARKs are deterministic

Converting an unchanged store twice mints the same identifiers. That is what
lets you re-export after adding runs without churning the ids of everything
that was already there — the crate can be updated instead of replaced.

(ARK sources for MLflow deliberately *include* the run id: unlike a workflow
re-run, re-training is a genuinely different event and deserves its own
identity.)

In [9]:
import tempfile

with tempfile.TemporaryDirectory() as elsewhere:
    again = mlflow_conversion.convert(
        "import", MLRUNS,
        experiment="iris-classifier",
        crate_dir=elsewhere,             # a different destination, on purpose
        naan="59853",
        name="Iris classifier — MLflow experiment",
        schemas=True,
    )

first = [node["@id"] for node in crate["@graph"]]
second = [node["@id"] for node in again["@graph"]]
print(f"same identifiers: {first == second}")
print(f"all unique:       {len(first) == len(set(first))}")

same identifiers: True
all unique:       True


## 8. What is on disk

`copy_artifacts=True` made the crate self-contained: the metadata references
its files by crate-relative `contentUrl`, and those files are right there. You
can zip this directory and hand it to someone.

In [10]:
for path in sorted(CRATE.rglob("*")):
    if path.is_file():
        size = path.stat().st_size
        print(f"{size:>9,}  {path.relative_to(CRATE)}")

      170  feature-importance-0742299e/feature_importance.csv
       27  rf-baseline-2dcf075c/confusion_matrix.csv
    1,126  rf-baseline-2dcf075c/model/MLmodel
      285  rf-baseline-2dcf075c/model/conda.yaml
      171  rf-baseline-2dcf075c/model/input_example.json
  282,253  rf-baseline-2dcf075c/model/model.pkl
      122  rf-baseline-2dcf075c/model/python_env.yaml
      152  rf-baseline-2dcf075c/model/requirements.txt
      382  rf-baseline-2dcf075c/model/serving_input_example.json
   13,612  ro-crate-metadata.json


## Where to go next

- **Another format?** Every converter in this package works the same way —
  see `examples/` next door for D4D datasheets, C2M2 datapackages, Workflow
  Run RO-Crates, Cromwell and Snakemake runs, and the Croissant export.
- **Croissant.** `crate → Croissant` is one more call
  (`examples/export_croissant.py`), which is how an MLflow experiment ends up
  loadable by an ML data ecosystem.
- **Don't want the artifacts copied?** `copy_artifacts=False` leaves the files
  where MLflow put them and references them by `localPath`.
- **Scoping.** `run_id="…"` exports one run and its nested children instead of
  a whole experiment.
- **How the mapping is defined.** `plugins/mlflow/entities.csv` and
  `properties.csv` — every field above is one row in one of those two files.